# 01 - Extração Distribuída no Cluster Spark & EDA Consolidado
**Squad 2 — Real Time for Business | Unificação Dupla 1 & Dupla 5**  
**Integrantes:** Zaiden Emiliano Segundo Seleme & Lucas Sousa Santos Oliveira  
**Tabelas de Escopo:**
1. `ecommerce_produtos` (Grupo 1)
2. `ecommerce_categorias` (Grupo 1)
3. `ecommerce_rastreamento_entregas` (Grupo 5)
4. `ecommerce_enderecos` (Grupo 5)

**Branch:** `feat/squad2-dupla1e5`

---

### Objetivos:
1. Realizar a leitura distribuída e paralela de todos os micro-lotes `.parquet` no contêiner `raw/real-time-data/` para as 4 tabelas.
2. Contornar particularidades de schema no Databricks Serverless (`TIMESTAMP_NANOS` em rastreamento e `INT32 Null` em endereços).
3. Deduplicação distribuída por chave primária (`sku`, `id_categoria`, `id_rastreamento`, `id_endereco`).
4. Análise Exploratória de Dados (EDA) cruzada: catálogo de produtos, integridade da hierarquia de categorias, volumetria de status logísticos e dispersão geográfica de clientes.

In [ ]:
# Instalação de dependências no escopo da sessão Serverless
%pip install python-dotenv==1.0.1 azure-storage-file-datalake azure-identity pandas pyarrow --quiet

In [ ]:
dbutils.library.restartPython()

In [ ]:
import os
import io
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from azure.storage.filedatalake import DataLakeServiceClient
from azure.identity import ClientSecretCredential
from dotenv import load_dotenv, find_dotenv
from pyspark.sql.functions import col, countDistinct, count

dotenv_path = find_dotenv()
if not dotenv_path:
    candidatos = [
        os.path.join(os.getcwd(), ".env"),
        os.path.join(os.path.dirname(os.getcwd()), ".env"),
        os.path.join(os.path.dirname(os.path.dirname(os.getcwd())), ".env")
    ]
    for c in candidatos:
        if os.path.exists(c):
            dotenv_path = c
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

assert client_id and tenant_id and client_secret, "Credenciais do Azure ADLS não encontradas no .env"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

print("Ambiente configurado com sucesso para extração e EDA!")

from pyspark.sql.types import (
    StructType, StructField, StringType, LongType, DoubleType, BooleanType, TimestampType
)

schema_spark_produtos = StructType([
    StructField("sku", StringType(), True),
    StructField("nome_produto", StringType(), True),
    StructField("descricao", StringType(), True),
    StructField("id_categoria", LongType(), True),
    StructField("preco_lista", DoubleType(), True),
    StructField("unidade_medida", StringType(), True),
    StructField("nome_marca", StringType(), True),
    StructField("is_ativo", BooleanType(), True)
])

schema_spark_categorias = StructType([
    StructField("id_categoria", LongType(), True),
    StructField("nome_categoria", StringType(), True),
    StructField("id_categoria_pai", LongType(), True),
    StructField("nome_categoria_pai", StringType(), True),
    StructField("tipo_categoria", StringType(), True)
])

schema_spark_rastreamento = StructType([
    StructField("id_rastreamento", LongType(), True),
    StructField("id_pedido_ecommerce", LongType(), True),
    StructField("codigo_rastreio", StringType(), True),
    StructField("id_transportadora", LongType(), True),
    StructField("nome_transportadora", StringType(), True),
    StructField("status_entrega", StringType(), True),
    StructField("dt_evento", TimestampType(), True),
    StructField("observacao", StringType(), True)
])

schema_spark_enderecos = StructType([
    StructField("id_endereco", LongType(), True),
    StructField("id_cliente", LongType(), True),
    StructField("logradouro", StringType(), True),
    StructField("numero", LongType(), True),
    StructField("complemento", StringType(), True),
    StructField("bairro", StringType(), True),
    StructField("cep", StringType(), True),
    StructField("cidade", StringType(), True),
    StructField("estado", StringType(), True),
    StructField("latitude", DoubleType(), True),
    StructField("longitude", DoubleType(), True),
    StructField("apelido", StringType(), True),
    StructField("is_principal", BooleanType(), True)
])

schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_seguro_spark(caminho, schema_spark):
    """Lê arquivos Parquet do RAW tratando ausência de arquivos sem quebrar o Spark."""
    try:
        df = spark.read.options(**adls_options).parquet(caminho)
        if df.limit(1).count() == 0:
            print(f"  [AVISO] Caminho existe mas não contém dados: '{caminho}'.")
            return spark.createDataFrame([], schema_spark)
        return df
    except Exception as e:
        print(f"  [AVISO] Nenhum arquivo encontrado no RAW em '{caminho}'. ({e.__class__.__name__})")
        print("  -> Retornando DataFrame vazio com schema tipado para garantir resiliência da esteira.")
        return spark.createDataFrame([], schema_spark)

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa, schema_spark):
    """Lê micro-lotes do RAW com PyArrow tratando ausência de arquivos sem quebrar o interpretador."""
    try:
        cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
        service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
        fs = service.get_file_system_client("raw")
        
        paths = fs.get_paths(path="real-time-data")
        arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
        
        if not arquivos:
            print(f"  [AVISO] Nenhum arquivo correspondente a '{nome_arquivo_alvo}' encontrado em 'raw/real-time-data/'.")
            print("  -> Retornando DataFrame vazio com schema tipado para continuidade da esteira.")
            return spark.createDataFrame([], schema_spark)

        tables = []
        for p in arquivos:
            fc = fs.get_file_client(p)
            data = fc.download_file().readall()
            t = pq.read_table(io.BytesIO(data))
            if t.num_rows == 0 or len(t.schema.names) == 0:
                continue
            if set(schema_pa.names).issubset(set(t.schema.names)):
                t = t.select(schema_pa.names)
            t_casted = t.cast(schema_pa)
            full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
            t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
            tables.append(t_with_meta)
            
        if not tables:
            print(f"  [AVISO] Arquivos encontrados para '{nome_arquivo_alvo}' estavam vazios.")
            return spark.createDataFrame([], schema_spark)
        combined = pa.concat_tables(tables)
        return spark.createDataFrame(combined.to_pandas())
    except Exception as e:
        print(f"  [AVISO] Falha ao ler '{nome_arquivo_alvo}' no RAW: {e}")
        return spark.createDataFrame([], schema_spark)

print("Funções de leitura segura com proteção contra ausência de arquivos prontas!")

In [ ]:
schema_rastreamento_pa = pa.schema([
    ('id_rastreamento', pa.int64()),
    ('id_pedido_ecommerce', pa.int64()),
    ('codigo_rastreio', pa.string()),
    ('id_transportadora', pa.int64()),
    ('nome_transportadora', pa.string()),
    ('status_entrega', pa.string()),
    ('dt_evento', pa.timestamp('us')),
    ('observacao', pa.string()),
])

schema_enderecos_pa = pa.schema([
    ('id_endereco', pa.int64()),
    ('id_cliente', pa.int64()),
    ('logradouro', pa.string()),
    ('numero', pa.int64()),
    ('complemento', pa.string()),
    ('bairro', pa.string()),
    ('cep', pa.string()),
    ('cidade', pa.string()),
    ('estado', pa.string()),
    ('latitude', pa.float64()),
    ('longitude', pa.float64()),
    ('apelido', pa.string()),
    ('is_principal', pa.bool_()),
])

def ler_parquet_raw_resiliente(nome_arquivo_alvo, schema_pa):
    cred = ClientSecretCredential(tenant_id=tenant_id, client_id=client_id, client_secret=client_secret)
    service = DataLakeServiceClient(account_url=f"https://{storage_account}.dfs.core.windows.net", credential=cred)
    fs = service.get_file_system_client("raw")
    
    paths = fs.get_paths(path="real-time-data")
    arquivos = [p.name for p in paths if p.name.endswith(nome_arquivo_alvo)]
    
    tables = []
    for p in arquivos:
        fc = fs.get_file_client(p)
        data = fc.download_file().readall()
        t = pq.read_table(io.BytesIO(data))
        if t.num_rows == 0 or len(t.schema.names) == 0:
            continue
        if set(schema_pa.names).issubset(set(t.schema.names)):
            t = t.select(schema_pa.names)
        t_casted = t.cast(schema_pa)
        full_path = f"abfss://raw@{storage_account}.dfs.core.windows.net/{p}"
        t_with_meta = t_casted.append_column("bronze_source_file", pa.array([full_path] * len(t_casted)))
        tables.append(t_with_meta)
        
    if not tables:
        return None
    combined = pa.concat_tables(tables)
    return spark.createDataFrame(combined.to_pandas())

print("Schemas e funções resilientes prontas.")

base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
caminho_produtos = f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet"
caminho_categorias = f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet"

print(f"Lendo produtos de forma segura em: {caminho_produtos}")
df_produtos_raw = ler_parquet_raw_seguro_spark(caminho_produtos, schema_spark_produtos)
total_bruto_prod = df_produtos_raw.count()

print(f"Lendo categorias de forma segura em: {caminho_categorias}")
df_categorias_raw = ler_parquet_raw_seguro_spark(caminho_categorias, schema_spark_categorias)
total_bruto_cat = df_categorias_raw.count()

# Deduplicação distribuída por PK (mesmo vazio, funciona perfeitamente)
df_produtos = df_produtos_raw.dropDuplicates(["sku"])
total_unicos_prod = df_produtos.count()

df_categorias = df_categorias_raw.dropDuplicates(["id_categoria"])
total_unicos_cat = df_categorias.count()

print("\n=== Consolidação Grupo 1 ===")
print(f"Produtos:   {total_bruto_prod} linhas brutas -> {total_unicos_prod} SKUs únicos")
print(f"Categorias: {total_bruto_cat} linhas brutas -> {total_unicos_cat} Categorias únicas")

if total_unicos_prod > 0:
    display(df_produtos.limit(5))
if total_unicos_cat > 0:
    display(df_categorias.limit(5))

In [ ]:
base_raw = f"abfss://raw@{storage_account}.dfs.core.windows.net/real-time-data"
caminho_produtos = f"{base_raw}/*/*/*/*/ecommerce_produtos.parquet"
caminho_categorias = f"{base_raw}/*/*/*/*/ecommerce_categorias.parquet"

print(f"Lendo produtos de: {caminho_produtos}")
df_produtos_raw = spark.read.options(**adls_options).parquet(caminho_produtos)
total_bruto_prod = df_produtos_raw.count()

print(f"Lendo categorias de: {caminho_categorias}")
df_categorias_raw = spark.read.options(**adls_options).parquet(caminho_categorias)
total_bruto_cat = df_categorias_raw.count()

# Deduplicação distribuída por PK
df_produtos = df_produtos_raw.dropDuplicates(["sku"])
total_unicos_prod = df_produtos.count()

df_categorias = df_categorias_raw.dropDuplicates(["id_categoria"])
total_unicos_cat = df_categorias.count()

print("\n=== Consolidação Grupo 1 ===")
print(f"Produtos:   {total_bruto_prod} linhas brutas -> {total_unicos_prod} SKUs únicos")
print(f"Categorias: {total_bruto_cat} linhas brutas -> {total_unicos_cat} Categorias únicas")

display(df_produtos.limit(5))
display(df_categorias.limit(5))

print("Lendo rastreamento e endereços de forma segura via Leitor Resiliente...")
df_rastreamento_raw = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa, schema_spark_rastreamento)
df_enderecos_raw = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa, schema_spark_enderecos)

total_bruto_rast = df_rastreamento_raw.count()
total_bruto_end = df_enderecos_raw.count()

# Deduplicação distribuída por PK
df_rastreamento = df_rastreamento_raw.dropDuplicates(["id_rastreamento"])
total_unicos_rast = df_rastreamento.count()

df_enderecos = df_enderecos_raw.dropDuplicates(["id_endereco"])
total_unicos_end = df_enderecos.count()

print("\n=== Consolidação Grupo 5 ===")
print(f"Rastreamento: {total_bruto_rast} linhas brutas -> {total_unicos_rast} Eventos únicos")
print(f"Endereços:    {total_bruto_end} linhas brutas -> {total_unicos_end} Endereços únicos")

if total_unicos_rast > 0:
    display(df_rastreamento.limit(5))
if total_unicos_end > 0:
    display(df_enderecos.limit(5))

In [ ]:
print("Lendo rastreamento e endereços via Leitor Resiliente Serverless...")
df_rastreamento_raw = ler_parquet_raw_resiliente("ecommerce_rastreamento.parquet", schema_rastreamento_pa)
df_enderecos_raw = ler_parquet_raw_resiliente("ecommerce_enderecos.parquet", schema_enderecos_pa)

total_bruto_rast = df_rastreamento_raw.count()
total_bruto_end = df_enderecos_raw.count()

# Deduplicação distribuída por PK
df_rastreamento = df_rastreamento_raw.dropDuplicates(["id_rastreamento"])
total_unicos_rast = df_rastreamento.count()

df_enderecos = df_enderecos_raw.dropDuplicates(["id_endereco"])
total_unicos_end = df_enderecos.count()

print("\n=== Consolidação Grupo 5 ===")
print(f"Rastreamento: {total_bruto_rast} linhas brutas -> {total_unicos_rast} Eventos únicos")
print(f"Endereços:    {total_bruto_end} linhas brutas -> {total_unicos_end} Endereços únicos")

display(df_rastreamento.limit(5))
display(df_enderecos.limit(5))

if total_unicos_rast > 0:
    print("=== 1. Distribuição de Status Logísticos de Entrega ===")
    df_rastreamento.groupBy("status_entrega").agg(count("*").alias("total")).orderBy(col("total").desc()).show(truncate=False)
else:
    print("=== 1. Distribuição de Status Logísticos: [PULADO - SEM DADOS EM RAW] ===")

if total_unicos_end > 0:
    print("=== 2. Distribuição Geográfica de Endereços por Estado - Top 10 ===")
    df_enderecos.groupBy("estado").agg(count("*").alias("total")).orderBy(col("total").desc()).show(10, truncate=False)
else:
    print("=== 2. Distribuição Geográfica: [PULADO - SEM DADOS EM RAW] ===")

if total_unicos_prod > 0 and total_unicos_cat > 0:
    print("=== 3. Integridade Referencial Produtos -> Categorias ===")
    prods_sem_cat = df_produtos.join(df_categorias, "id_categoria", "left_anti").count()
    print(f"Produtos sem categoria correspondente (órfãos): {prods_sem_cat} (Status: {'[100% ÍNTEGRO]' if prods_sem_cat == 0 else '[ALERTA: ÓRFÃOS]'})")
    print("=== 4. Distribuição de Preço dos Produtos ===")
    df_produtos.selectExpr("min(preco_lista) as preco_min", "avg(preco_lista) as preco_medio", "max(preco_lista) as preco_max").show()
else:
    print("=== 3/4. Integridade de Produtos/Categorias: [PULADO - SEM DADOS EM RAW] ===")

print("\nExtração e Análise Exploratória concluídas com tratamento defensivo de RAW!")

In [ ]:
print("=== 1. Distribuição de Status Logísticos de Entrega (Grupo 5) ===")
df_rastreamento.groupBy("status_entrega").agg(count("*").alias("total")).orderBy(col("total").desc()).show(truncate=False)

print("=== 2. Distribuição Geográfica de Endereços por Estado - Top 10 (Grupo 5) ===")
df_enderecos.groupBy("estado").agg(count("*").alias("total")).orderBy(col("total").desc()).show(10, truncate=False)

print("=== 3. Integridade Referencial Produtos -> Categorias (Grupo 1) ===")
prods_sem_cat = df_produtos.join(df_categorias, "id_categoria", "left_anti").count()
print(f"Produtos sem categoria correspondente (órfãos): {prods_sem_cat} (Status: {'[100% ÍNTEGRO]' if prods_sem_cat == 0 else '[ALERTA: ÓRFÃOS]'})")

print("=== 4. Distribuição de Preço dos Produtos (Grupo 1) ===")
df_produtos.selectExpr("min(preco_lista) as preco_min", "avg(preco_lista) as preco_medio", "max(preco_lista) as preco_max").show()

print("\nExtração e Análise Exploratória unificada concluídas com sucesso!")